# 세부 시나리오 MCP 파이프라인 플레이그라운드

선택된 `detail_scenario_code` 하나를 기준으로 **입력 arguments → MCP 호출/페이지 반복 → 결과 전처리 → 프런트용 formatted_result** 전체를 외부 서버 없이 재현합니다.

실제 수정 위치는 `app/mcp/scenarios/<agent>.py`의 async handler와 `*_output()` 함수입니다. 이 노트북의 executor 응답과 사용자 정의 함수를 먼저 바꿔 본 뒤 운영 코드에 반영하세요.

In [1]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import sys
from typing import Any
from pprint import pprint

# Jupyter는 보통 이 notebook이 있는 notebooks/를 현재 폴더로 사용한다.
# 위로 올라가며 app/ 폴더를 찾아 프로젝트 import 경로에 추가한다.
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'app').is_dir():
        sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError('프로젝트 루트(app 폴더)를 찾지 못했습니다.')

from app.mcp.models import McpExecutionResult
from app.mcp.result_adapters import adapt_mcp_result
from app.mcp.scenario_runtime import ScenarioMcpHandlerContext
from app.mcp.scenarios.contracts import ScenarioMcpOutput, ScenarioMcpOutputContext
from app.mcp.scenarios.registry import (
    ScenarioMcpHandlerSpec,
    get_scenario_handler_spec,
    run_scenario_handler,
)
from app.renderables import ScenarioAnswer, create_table_renderable, extract_data_items
from app.subagents.models import SubagentResult

# 이 셀의 값만 바꾸면 다른 등록 detail을 재현할 수 있습니다.
AGENT_CODE = 'PERFORMANCE_FEE'
DETAIL_CODE = 'COMPOSITE_CONVERSION_EXCLUDED'

def selected_subagent(detail_code: str = DETAIL_CODE) -> SubagentResult:
    return SubagentResult.model_validate({
        'agent_code': AGENT_CODE,
        'prompt_version': 'notebook',
        'scenario_code': 'COMPOSITE_CONVERSION',
        'scenario_name': '복합환산조회',
        'detail_scenario_code': detail_code,
        'detail_scenario_name': detail_code,
        'parameters': {'closing_year_month': '202608', 'reference_date': ''},
    })

def show(value: Any) -> None:
    pprint(value, sort_dicts=False, width=120)


## 1. 페이지 MCP 응답을 흉내 내는 executor

첫 호출은 일부 `nextkey`와 `gridct`를, 두 번째 호출은 모든 `nextkey`가 빈 결과를 반환합니다. 실제 MCP schema를 받은 뒤에는 `page_data()`만 그 구조로 바꿔 테스트하면 됩니다.

In [2]:
class ScriptedPagingExecutor:
    def __init__(self) -> None:
        self.calls: list[dict[str, Any]] = []

    def page_data(self, page_number: int) -> list[dict[str, Any]]:
        # TODO: 실제 MCP structuredContent.data의 objId/objVal 구조로 교체
        has_next = page_number == 1
        return [
            {'objId': 'column1', 'objVal': str(100 * page_number)},
            {'objId': 'column2', 'objVal': f'{page_number + 1}건'},
            {'objId': 'no1Grid', 'objVal': [
                [
                    {'objId': 'customerId', 'objVal': f'C-{page_number}-1'},
                    {'objId': 'amount', 'objVal': page_number * 10},
                ],
                [
                    {'objId': 'customerId', 'objVal': f'C-{page_number}-2'},
                    {'objId': 'amount', 'objVal': page_number * 20},
                ],
            ]},
            {'objId': 'firstNextKey', 'objVal': 'NEXT-1' if has_next else ''},
            {'objId': 'secondNextKey', 'objVal': ''},
            {'objId': 'gridct', 'objVal': 50},
        ]

    async def execute(self, **kwargs: Any) -> McpExecutionResult:
        arguments = dict(kwargs['argument_overrides'])
        self.calls.append({
            'arguments': arguments,
            'request_context': dict(kwargs['request_context']),
        })
        page_number = len(self.calls)
        return McpExecutionResult(
            backend='notebook',
            tool_name='test_tool',
            request_id=f'notebook-page-{page_number}',
            arguments=arguments,
            succeeded=True,
            result={'data': self.page_data(page_number)},
        )

async def run_selected_scenario(
    spec: ScenarioMcpHandlerSpec,
    executor: Any,
    subagent: SubagentResult | None = None,
):
    subagent = subagent or selected_subagent()
    context = ScenarioMcpHandlerContext(
        handler_code=spec.code,
        executor=executor,
        subagent=subagent,
        employee_id='K3003980',
        session_id='notebook-session',
        thread_id='notebook-thread',
        request_context={
            'access_token': 'notebook-access-token',
            'recruitment_org_type_code': '12',
            'user': {'id': 'K3003980', 'deptcode': 'D01', 'deptname': '개발팀'},
        },
    )
    outcome = await run_scenario_handler(spec=spec, context=context)
    terminal = adapt_mcp_result(
        execution=outcome.terminal,
        subagent=subagent,
        employee_id=context.employee_id,
        session_id=context.session_id,
        thread_id=context.thread_id,
        request_context=context.request_context,
        workflow_results=outcome.results,
    )
    return context, outcome, terminal


## 2. 실제 등록 handler를 처음부터 끝까지 실행

이 셀은 registry에서 실제 `COMPOSITE_CONVERSION_SCORE` handler와 output handler를 가져옵니다. 두 번째 MCP 호출 arguments에 `firstNextKey`, 빈 `secondNextKey`, `no1PgeSize=50`이 모두 들어가는지 확인하세요.

In [3]:
spec = get_scenario_handler_spec(AGENT_CODE, DETAIL_CODE)
assert spec is not None
executor = ScriptedPagingExecutor()
context, outcome, terminal = await run_selected_scenario(spec, executor)

print('handler code:', spec.code)
print('output handler:', spec.output_handler_code)
print('MCP 호출 수:', len(executor.calls))
show(executor.calls)

print('\n--- workflow 원장 ---')
show([item.model_dump(mode='json') for item in outcome.results])

print('\n--- 프런트 전달용 formatted_result ---')
show(terminal.formatted_result)


handler code: performance_fee.composite_conversion_excluded.v1
output handler: performance_fee.composite_conversion_excluded_output.v1
MCP 호출 수: 2
[{'arguments': {'param1': '202608', 'param2': 'K3003980'},
  'request_context': {'access_token': 'notebook-access-token',
                      'recruitment_org_type_code': '12',
                      'user': {'id': 'K3003980', 'deptcode': 'D01', 'deptname': '개발팀'}}},
 {'arguments': {'param1': '202608',
                'param2': 'K3003980',
                'firstNextKey': 'NEXT-1',
                'secondNextKey': '',
                'no1PgeSize': 50},
  'request_context': {'access_token': 'notebook-access-token',
                      'recruitment_org_type_code': '12',
                      'user': {'id': 'K3003980', 'deptcode': 'D01', 'deptname': '개발팀'}}}]

--- workflow 원장 ---
[{'backend': 'notebook',
  'tool_name': 'test_tool',
  'request_id': 'notebook-page-1',
  'arguments': {'param1': '202608', 'param2': 'K3003980'},
  'succeeded': Tru

## 3. `no1Grid`와 중복 컬럼을 원하는 형태로 전처리

아래 함수는 운영 코드를 바꾸지 않고 이 노트북에서만 output handler를 교체합니다. 각 페이지의 `no1Grid` 리스트를 하나의 행 목록으로 평탄화합니다. 확인 후 같은 함수를 `performance_fee.py`의 `*_output()`으로 옮기고 registry에 연결하면 됩니다.

In [4]:
def _field_rows(value: Any) -> list[list[dict[str, Any]]]:
    if not isinstance(value, list):
        return []
    if value and all(isinstance(item, dict) and 'objId' in item for item in value):
        return [value]
    rows = []
    for child in value:
        rows.extend(_field_rows(child))
    return rows

def flattened_grid_output(context: ScenarioMcpOutputContext) -> ScenarioMcpOutput:
    rows = []
    for item in context.data_items():
        if str(item.get('objId', '')).strip() != 'no1Grid':
            continue
        page = int(item.get('_function_call', {}).get('index', 0)) + 1
        for field_items in _field_rows(item.get('objVal')):
            grid_row = {field.get('objId'): field.get('objVal') for field in field_items}
            rows.append({
                'page': page,
                'customerId': grid_row.get('customerId', ''),
                'amount': grid_row.get('amount', ''),
            })

    table_rows = [(row['page'], row['customerId'], row['amount']) for row in rows]
    return ScenarioMcpOutput(
        data={'gridRows': rows, 'pageCount': len(context.workflow.get('batches', []))},
        answer=ScenarioAnswer(
            text=f'[복합환산 Grid 결과]\n- 총 {len(rows)}행',
            renderables=[create_table_renderable(
                code='grid-table',
                title='복합환산 상세 Grid',
                format='markdown',
                columns=('페이지', '고객 ID', '금액'),
                rows=table_rows,
            )],
        ),
        metadata={'preprocessor': 'notebook.flattened_grid_output'},
    )

custom_spec = replace(
    spec,
    output_handler=flattened_grid_output,
    output_handler_code='notebook.flattened_grid_output.v1',
)
executor = ScriptedPagingExecutor()
_, _, terminal = await run_selected_scenario(custom_spec, executor)
show(terminal.formatted_result)


{'format': 'query.v1',
 'adapter_code': 'PERFORMANCE_FEE:COMPOSITE_CONVERSION_EXCLUDED:function',
 'result_formatter_code': 'performance_fee.composite_conversion_excluded_output.v1',
 'output_handler_code': 'performance_fee.composite_conversion_excluded_output.v1',
 'data': {'commonItems': [{'objId': 'column1',
                           'objVal': '100',
                           '_function_call': {'index': 0,
                                              'requestId': 'notebook-page-1',
                                              'arguments': {'param1': '202608', 'param2': 'K3003980'}}},
                          {'objId': 'column2',
                           'objVal': '2건',
                           '_function_call': {'index': 0,
                                              'requestId': 'notebook-page-1',
                                              'arguments': {'param1': '202608', 'param2': 'K3003980'}}}],
          'gridRows': [{'_page': 1, 'customerId': 'C-1-1', 'amount': 1

## 4. 첫 MCP 결과를 사용해 두 번째 MCP를 여러 번 호출하는 예시

`context.call_many()`에 전달하는 `arguments_list`를 일반 Python으로 만들면, 1→N·N→N·조건부 호출 모두 detail 함수 안에서 자유롭게 구성할 수 있습니다.

In [5]:
class TwoStageExecutor:
    def __init__(self):
        self.calls = []

    async def execute(self, **kwargs):
        arguments = dict(kwargs['argument_overrides'])
        tool_name = kwargs['subagent'].mcp_workflow.steps[0].tool.name
        self.calls.append({'tool': tool_name, 'arguments': arguments})
        if tool_name == 'target_tool':
            data = [
                {'objId': 'customerId', 'objVal': 'C-001'},
                {'objId': 'customerId', 'objVal': 'C-002'},
            ]
        else:
            customer_id = arguments['customerId']
            data = [{'objId': 'detail', 'objVal': {'customerId': customer_id, 'status': 'OK'}}]
        return McpExecutionResult(
            backend='notebook', tool_name=tool_name, request_id=f'call-{len(self.calls)}',
            arguments=arguments, succeeded=True, result={'data': data},
        )

async def target_then_details(context: ScenarioMcpHandlerContext) -> McpExecutionResult:
    targets = await context.call(
        step_code='TARGETS', tool_name='target_tool', arguments={'employeeId': context.employee_id}
    )
    detail_arguments = [
        {'customerId': item['objVal'], 'requestContextExample': context.request_context.get('endpoint', '')}
        for item in extract_data_items(targets.result)
        if item.get('objId') == 'customerId'
    ]
    return await context.call_many(
        step_code='DETAILS', tool_name='detail_tool', arguments_list=detail_arguments,
        error_policy='continue', max_items=100,
    )

def two_stage_output(context: ScenarioMcpOutputContext) -> ScenarioMcpOutput:
    details = [item['objVal'] for item in context.data_items() if item.get('objId') == 'detail']
    rows = [(value.get('customerId', ''), value.get('status', '')) for value in details]
    return ScenarioMcpOutput(
        data={'details': details},
        answer=ScenarioAnswer(
            text=f'[상세 조회] {len(rows)}건',
            renderables=[create_table_renderable(
                code='details-table', title='상세 결과', format='markdown',
                columns=('고객 ID', '상태'), rows=rows,
            )],
        ),
    )

two_stage_spec = ScenarioMcpHandlerSpec(
    code='notebook.target_then_details.v1',
    handler=target_then_details,
    output_handler=two_stage_output,
    output_handler_code='notebook.two_stage_output.v1',
)
executor = TwoStageExecutor()
_, outcome, terminal = await run_selected_scenario(two_stage_spec, executor)
show(executor.calls)
show(terminal.formatted_result)


[{'tool': 'target_tool', 'arguments': {'employeeId': 'K3003980'}},
 {'tool': 'detail_tool', 'arguments': {'customerId': 'C-001', 'requestContextExample': ''}},
 {'tool': 'detail_tool', 'arguments': {'customerId': 'C-002', 'requestContextExample': ''}}]
{'format': 'query.v1',
 'adapter_code': 'PERFORMANCE_FEE:COMPOSITE_CONVERSION_EXCLUDED:function',
 'result_formatter_code': 'performance_fee.composite_conversion_excluded_output.v1',
 'output_handler_code': 'performance_fee.composite_conversion_excluded_output.v1',
 'data': {'commonItems': [{'objId': 'detail',
                           'objVal': {'customerId': 'C-001', 'status': 'OK'},
                           '_function_call': {'index': 0,
                                              'requestId': 'call-2',
                                              'arguments': {'customerId': 'C-001', 'requestContextExample': ''}}}],
          'gridRows': [],
          'pageCount': 0},
 'parameters': {'closing_year_month': '202608', 'reference_da

## 5. 오류 흐름 테스트

아래처럼 nextkey가 있는데 `gridct`가 누락된 응답을 만들면 `_all_next_key_arguments()`의 계약 오류가 handler 안전 결과로 변환됩니다. 실제 사용자 화면에는 내부 예외 대신 안전한 오류 문구가 전달되는지 확인할 수 있습니다.

테스트 순서: `page_data()`의 `gridct`를 삭제 → 2번 셀 실행 → `outcome.terminal.error`와 `user_message` 확인 → 실제 schema에 맞게 handler callback 수정.

## 수정 체크리스트

1. 실제 MCP 응답을 `ScriptedPagingExecutor.page_data()`에 붙여 넣는다.
2. `app/mcp/scenarios/<agent>.py`의 async handler에서 tool/arguments/호출 흐름을 수정한다.
3. 같은 파일의 `*_output()`에서 raw result, `data_items()`, `workflow['batches']`를 원하는 구조로 전처리한다.
4. `ScenarioMcpOutput.data`는 tester용 자유 형식 데이터, `ScenarioAnswer`는 프런트 답변/표다.
5. `/mock/intent-tester`에서 handlerCode, outputHandlerCode, preprocessedData, raw result를 재확인한다.

상세 설명은 `docs/15_FUNCTION_FIRST_MCP_INPUT_OUTPUT.md`를 참고하세요.